In [1]:
# 파일명: run_clip_reranking.py

import pandas as pd
import torch
from tqdm import tqdm

# CLIP 모델 관련 import
from transformers import AutoProcessor as CLIP_AutoProcessor, AutoModel as CLIP_AutoModel

# --- 1. 설정 (Configuration) ---
print("--- [단계 2/2] CLIP 재랭킹 및 최종 제출 시작 ---")
print("1. 설정을 초기화합니다...")

# 경로 설정
# [입력] 1단계에서 생성된 중간 결과 파일
INTERMEDIATE_CSV_PATH = "beit3_candidates.csv" 
# [입력] 최종 제출 형식을 위한 샘플 파일
SAMPLE_SUBMISSION_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv"
# [출력] 최종 결과 파일
FINAL_SUBMISSION_PATH = "beit3_clip_rerank_submission.csv"

# CLIP 모델 이름
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {DEVICE}")

# --- 2. CLIP 모델 로드 ---
print("\n2. CLIP 모델을 로드합니다...")
clip_processor = CLIP_AutoProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIP_AutoModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_model.eval()
print("CLIP 모델 로드 완료.")

# --- 3. 데이터 로드 ---
print(f"\n3. {INTERMEDIATE_CSV_PATH} 파일과 제출 샘플 파일을 로드합니다...")
try:
    df_candidates = pd.read_csv(INTERMEDIATE_CSV_PATH)
    df_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
except FileNotFoundError as e:
    print(f"오류: 필요한 파일을 찾을 수 없습니다: {e}")
    exit()

# --- 4. CLIP 재랭킹 루프 ---
print("\n4. CLIP 재랭킹을 시작합니다...")
index_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}

for index, row in tqdm(df_candidates.iterrows(), total=df_candidates.shape[0], desc="CLIP 재랭킹 진행률"):
    test_id = row['ID']
    candidate_answer = row['candidate_answer']
    choices = [row['A'], row['B'], row['C'], row['D']]

    # BEiT-3가 에러를 반환한 경우 건너뛰기
    if candidate_answer == 'Error':
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error'
        continue

    try:
        # CLIP 입력: BEiT-3의 예측(후보 답변) + 4개의 보기
        texts_for_clip = [str(candidate_answer)] + [str(c) for c in choices]
        
        inputs_clip = clip_processor(text=texts_for_clip, return_tensors="pt", padding=True)
        inputs_clip = {k: v.to(DEVICE) for k, v in inputs_clip.items()}
        
        with torch.no_grad():
            text_features = clip_model.get_text_features(**inputs_clip)
        
        # 코사인 유사도 계산
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        candidate_embedding = text_features[0]
        choice_embeddings = text_features[1:]
        
        similarity_scores = (candidate_embedding @ choice_embeddings.T).cpu()
        
        final_predicted_index = similarity_scores.argmax().item()
        prediction_label = index_to_label[final_predicted_index]
        
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = prediction_label

    except Exception as e:
        print(f"\n오류: ID '{test_id}' 재랭킹 중 예외 발생: {e}")
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error'

# --- 5. 결과 저장 및 출력 ---
df_submission.to_csv(FINAL_SUBMISSION_PATH, index=False)
print("\n" + "="*50)
print("모든 프로세스가 완료되었습니다.")
print(f"최종 결과가 {FINAL_SUBMISSION_PATH} 파일에 저장되었습니다.")
print("\n최종 제출 파일 미리보기 (상위 5개):")
print(df_submission.head())
print("="*50)

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- [단계 2/2] CLIP 재랭킹 및 최종 제출 시작 ---
1. 설정을 초기화합니다...
사용 디바이스: cuda

2. CLIP 모델을 로드합니다...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIP 모델 로드 완료.

3. beit3_candidates.csv 파일과 제출 샘플 파일을 로드합니다...

4. CLIP 재랭킹을 시작합니다...


CLIP 재랭킹 진행률: 100%|██████████| 60/60 [00:00<00:00, 77.65it/s]


모든 프로세스가 완료되었습니다.
최종 결과가 beit3_clip_rerank_submission.csv 파일에 저장되었습니다.

최종 제출 파일 미리보기 (상위 5개):
         ID answer
0  TEST_000      B
1  TEST_001      B
2  TEST_002      A
3  TEST_003      B
4  TEST_004      A
